# Modules over the triplet algebra $\mathcal W(2)$

The triplet algebras $\mathcal W(p)$ (Kausch; Gaberdiel–Kausch) are the standard examples of vertex algebras that are $C_2$-cofinite but not rational: they have finitely many simple modules, but also indecomposable modules on which $L_0$ is not diagonalizable (logarithmic modules). For $p = 2$, $\mathcal W(2)$ is the even part of a pair of symplectic fermions $\chi^\pm$ at $c = -2$ (Kausch; Abe). It has four simple modules $\mathcal W(h)$, of lowest weights $h = 0, 1, -\frac18, \frac38$; $\mathcal W(-\frac18)$ and $\mathcal W(\frac38)$ are projective, while the projective cover of the vacuum module has the structure (Gaberdiel–Kausch)
$$\mathcal P(0) = \mathcal W(0) \to 2\,\mathcal W(1) \to \mathcal W(0).$$

This notebook computes all of this from the definition of $\mathcal W(2)$ as a subalgebra, with general constructions of the library:

1. $\mathcal W(2)$ as the subalgebra of symplectic fermions generated by $T, W^+, W^0, W^-$ (`V.subalgebra`);
2. its presentation by generators, OPEs and relations (`S.presentation()`): the OPEs written in the generators satisfy the Jacobi identity only modulo null fields;
3. its Zhu algebra (`P.zhu_algebra_up_to(W)`), from two-sided Gröbner bases in a free algebra, and the modules of the Zhu algebra, found by solving its relations: the four simple tops and a Jordan block;
4. the simple modules, as Dong–Li–Mason's universal modules $\bar M(U)$ of the simple tops (`P.induced_module_up_to(U, max_weight, max_depth)`) and their simple quotients;
5. the module $\mathcal W(0) \to 2\,\mathcal W(1)$, induced from the trivial top;
6. the projective cover $\mathcal P(0)$, induced from the Jordan block of $L_0$;
7. the stability of these results in the weight of the relations and the depth;
8. what is not there yet.

Everything is checked against the symplectic fermion Fock spaces: the untwisted sector gives $\mathcal W(0)$ and $\mathcal W(1)$ (its even and odd parts), the $\mathbb Z_2$-twisted sector $\mathcal W(-\frac18)$ and $\mathcal W(\frac38)$, character by character in the depth and the fermion charge.

**Prerequisites:** `06_subalgebras_ideals_quotients.ipynb`, `09_zhu_algebras.ipynb`, `10_modules.ipynb`, `11_fusion_and_extensions.ipynb`.

In [1]:
import vertex_algebras as va
from vertex_algebras import *
from vertex_algebras import UniversalVertexAlgebra, NormallyOrderedExpressions

## 1. $\mathcal W(2)$ inside symplectic fermions

Symplectic fermions: odd fields $\chi^\pm$ of weight 1 with $\chi^+(z)\chi^-(w) \sim (z-w)^{-2}$, conformal vector $T = {:}\chi^-\chi^+{:}$ ($c = -2$), graded by the charge ($\chi^\pm$ of charge $\pm1$). The triplet algebra is generated by $T$ and the weight-3 fields $W^+ = {:}\chi^+\partial\chi^+{:}$, $W^0 = {:}\chi^+\partial\chi^-{:} + {:}\chi^-\partial\chi^+{:}$, $W^- = {:}\chi^-\partial\chi^-{:}$ of charges $2, 0, -2$; the closure finds no further strong generators:

In [2]:
Y.<cp, cm> = NormallyOrderedExpressions(QQ, parities={'cp': 1, 'cm': 1})
SF = UniversalVertexAlgebra(QQ, Y, {('cp', 'cm'): [0, 1]}, conformal_vector=cm*cp,
                            horizontal_grading={'cp': 1, 'cm': -1})
a, b = SF.gens()
Wp, Wm, W0 = a.no(a.derivative()), b.no(b.derivative()), a.no(b.derivative()) + b.no(a.derivative())
W2 = SF.subalgebra([SF.conformal_vector(), Wp, W0, Wm], names=['T', 'Wp', 'W0', 'Wm'],
                   horizontal_grading='ambient', conformal_vector='ambient')
print(W2.variable_names(), [W2.degree(g) for g in W2.gens()], W2.central_charge())

('T', 'Wp', 'W0', 'Wm') [2, 3, 3, 3] -2


The characters of the four simple modules come from the Fock spaces of the fermion modes, $\prod_{n \ge 1}(1 + y q^n)(1 + y^{-1} q^n)$ (untwisted, modes $\chi^\pm_{-n}$) and $\prod_{n \ge 0}(1 + y q^{n+1/2})(1 + y^{-1} q^{n+1/2})$ (twisted, modes $\chi^\pm_{-n-1/2}$), $y$ counting the charge: the even and odd parts in the fermion number (= the parity of the charge). `chi(h, d, g)` is the dimension of $\mathcal W(h)$ at depth $d$ (weight $h + d$) and charge $g$. The vacuum module is $\mathcal W(2)$ itself:

In [3]:
R.<y> = LaurentPolynomialRing(QQ)
S.<s> = PowerSeriesRing(R, default_prec=20)          # s = q^(1/2)
untwisted = prod((1 + y*s^(2*n)) * (1 + y^-1*s^(2*n)) for n in range(1, 10))
twisted = prod((1 + y*s^(2*n+1)) * (1 + y^-1*s^(2*n+1)) for n in range(0, 10))


def chi(h, d, g):
    """The dimension of the simple module W(h) at depth d and charge g."""
    F, k, parity = {0: (untwisted, 2*d, 0), 1: (untwisted, 2*d + 2, 1),
                    -1/8: (twisted, 2*d, 0), 3/8: (twisted, 2*d + 1, 1)}[h]
    return F[k][g] if (g - parity) % 2 == 0 else 0


charges = range(-6, 7)
print([W2.weight_space(d).dimension() for d in range(9)])
print([sum(chi(0, d, g) for g in charges) for d in range(9)])

[1, 0, 1, 4, 5, 8, 10, 16, 22]
[1, 0, 1, 4, 5, 8, 10, 16, 22]


## 2. The presentation

$\mathcal W(2)$ is strongly generated by $T, W^\pm, W^0$, but not freely. `W2.presentation()` writes it as $F/R$: $F$ is the universal vertex algebra on these four fields with their OPEs written in their normally ordered monomials, $R$ the kernel of the realization $F \to \mathcal W(2)$. $F$ itself does not satisfy the Jacobi identity: its failures are null fields, all of which lie in $R$. The first relations are at weight 6, e.g. ${:}W^+W^+{:} = 0$:

In [4]:
P = W2.presentation(); F = P.ambient()
fails = F.jacobi_identity_failures()
print(len(fails), all(x in P.defining_ideal() for x in fails.values()))
print(P.relations(6, horizontal_grade=4))
print({w: {g: len(P.relations(w, horizontal_grade=g)) for g in range(-6, 7, 2) if P.relations(w, horizontal_grade=g)}
       for w in range(6, 9)})

66 True
[:Wp Wp:]
{6: {-4: 1, -2: 2, 0: 3, 2: 2, 4: 1}, 7: {-4: 1, -2: 3, 0: 4, 2: 3, 4: 1}, 8: {-4: 3, -2: 6, 0: 9, 2: 6, 4: 3}}


## 3. The Zhu algebra and its modules

The Zhu algebra of the presentation is the free algebra on $[T], [W^+], [W^0], [W^-]$ modulo the commutator relations and the images of the relations of weight $\le W$ (two-sided Gröbner bases, Letterplace); it maps onto $A(\mathcal W(2))$ and stabilizes at dimension 11 from $W = 7$ on (Adamović–Milas). $[\omega] = [T]$ has the minimal polynomial $x^2(x - 1)(x + \frac18)(x - \frac38)$: the four lowest weights, $0$ with multiplicity 2:

In [5]:
print([(W, P.zhu_algebra_up_to(W).dimension()) for W in range(5, 10)])
Z = P.zhu_algebra_up_to(8)
x = polygen(QQ, 'x')


def minimal_polynomial(Z, z, bound=12):
    """The minimal polynomial of the element z of the finite-dimensional algebra Z."""
    powers = [Z.one()]
    for k in range(1, bound + 1):
        powers.append(powers[-1] * z)
        keys = sorted({w for p in powers for w in p.support()})
        K = matrix(QQ, [[p.monomial_coefficients().get(w, 0) for w in keys] for p in powers]).left_kernel()
        if K.dimension():
            return sum(c * x^i for i, c in enumerate(K.basis()[0])).monic()


print(Z.gens(), minimal_polynomial(Z, Z.gen(0)).factor())

[(5, +Infinity), (6, 15), (7, 11), (8, 11), (9, 11)]
([T], [Wp], [W0], [Wm]) (x - 1) * (x - 3/8) * (x + 1/8) * x^2


The modules of the Zhu algebra are the possible top levels. Its Gröbner basis is public (`Z.groebner_basis()`, in the free algebra on $x_0, \dots, x_3$ = $[T], [W^+], [W^0], [W^-]$), so the small modules can be found by solving it on matrices with unknown entries. One-dimensional modules of charge 0 ($[W^\pm]$ act by 0): $h = 0$ and $h = -\frac18$. Two-dimensional modules on vectors of charge $\pm 1$ ($[W^+]$ raises, $[W^-]$ lowers the charge by 2): besides sums of one-dimensional ones, the doublets at $h = 1$ and $h = \frac38$:

In [6]:
G = Z.groebner_basis()


def evaluate(f, mats):
    """The element f of the free algebra evaluated on the matrices mats (x_i -> mats[i])."""
    out = 0 * mats[0]
    for m, c in f.monomial_coefficients().items():
        p = mats[0].parent().identity_matrix()
        for v, e in m:
            p = p * mats[int(str(v)[1:])]^e
        out += c * p
    return out


R1.<t, w> = QQ[]
print(R1.ideal([evaluate(f, [matrix(R1, [[t]]), matrix(R1, [[0]]), matrix(R1, [[w]]), matrix(R1, [[0]])])[0, 0]
                for f in G]).variety())
R2.<p, m, c, d, t1, t2> = QQ[]
mats = [matrix(R2, [[t1, 0], [0, t2]]), matrix(R2, [[0, p], [0, 0]]), matrix(R2, [[c, 0], [0, d]]),
        matrix(R2, [[0, 0], [m, 0]])]
I2 = R2.ideal([e for f in G for e in evaluate(f, mats).list()])
for Q in I2.minimal_associated_primes():
    print(Q.gens())

[{w: 0, t: 0}, {w: 0, t: -1/8}]
[p, m, c, d, t1, t2]
[t2, 8*t1 + 1, d, c, m, p]
[8*t2 + 1, t1, d, c, m, p]
[8*t2 + 1, 8*t1 + 1, d, c, m, p]
[p*m + 4, t2 - 1, t1 - 1, d + 2, c - 2]
[4*p*m + 1, 8*t2 - 3, 8*t1 - 3, 2*d + 1, 2*c - 1]


So the simple tops are (normalising $[W^+] = \begin{pmatrix} 0 & 1 \\ 0 & 0\end{pmatrix}$ on the doublets): $11 = 1 + 1 + 4 + 4 + 1$, the last dimension being the nilpotent part of $[T]$ at $h = 0$, a $2 \times 2$ Jordan block. Module grades: the charges.

In [7]:
Z0 = matrix(QQ, 2, 2)
tops = {
    0: Z.module({'T': 0, 'Wp': 0, 'W0': 0, 'Wm': 0}, horizontal_grades=[0]),
    -1/8: Z.module({'T': -1/8, 'Wp': 0, 'W0': 0, 'Wm': 0}, horizontal_grades=[0]),
    1: Z.module({'T': identity_matrix(QQ, 2), 'Wp': matrix([[0, 1], [0, 0]]), 'W0': diagonal_matrix([2, -2]),
                 'Wm': matrix([[0, 0], [-4, 0]])}, horizontal_grades=[1, -1]),
    3/8: Z.module({'T': 3/8*identity_matrix(QQ, 2), 'Wp': matrix([[0, 1], [0, 0]]),
                   'W0': diagonal_matrix([1/2, -1/2]), 'Wm': matrix([[0, 0], [-1/4, 0]])}, horizontal_grades=[1, -1]),
}
jordan = Z.module({'T': matrix([[0, 1], [0, 0]]), 'Wp': Z0, 'W0': Z0, 'Wm': Z0}, horizontal_grades=[0, 0])
print(Z.dimension(), [U.dimension() for U in tops.values()], jordan.dimension())

11 [1, 1, 2, 2] 2


## 4. The simple modules

`P.induced_module_up_to(U, max_weight, max_depth)` is the universal module $\bar M(U)$ with top level $U$ (Dong–Li–Mason): the module induced over $F$, modulo what the relations of weight $\le$ `max_weight` generate (and the defects of the commutator formula, since $F$ does not satisfy the Jacobi identity: here there are none), computed to depth `max_depth`. For the tops at $h = -\frac18, 1, \frac38$ it is already simple (its maximal submodule is zero) and has the character of $\mathcal W(h)$, charge by charge. For $h = 0$ its simple quotient is the vacuum module $\mathcal W(0) = \mathcal W(2)$:

In [8]:
M = {}
for h, U in tops.items():
    D = 6 if U.dimension() == 1 else 5
    M[h] = P.induced_module_up_to(U, 8, D)
    J = M[h].maximal_submodule()
    L = M[h].simple_quotient()
    ok = all(L.weight_space(h + d, horizontal_grade=g).dimension() == chi(h, d, g) for d in range(D + 1) for g in charges)
    print(h, [L.weight_space(h + d).dimension() for d in range(D + 1)], ok,
          'J:', [J.weight_space(h + d).dimension() for d in range(D + 1)])

0 [1, 0, 1, 4, 5, 8, 10] True J: [0, 4, 4, 4, 8, 12, 24]


-1/8 [1, 1, 4, 5, 9, 13, 21] True J: [0, 0, 0, 0, 0, 0, 0]


1 [2, 2, 2, 4, 6, 12] True J: [0, 0, 0, 0, 0, 0]


3/8 [2, 2, 4, 6, 12, 16] True J: [0, 0, 0, 0, 0, 0]


## 5. The module $\mathcal W(0) \to 2\,\mathcal W(1)$

From the trivial top, $\bar M(\mathbb C)$ is not simple: its maximal submodule has the character of $2\,\mathcal W(1)$, and it is generated by the four vectors of depth 1, $T_{-1} v$, $W^\pm_{-1} v$, $W^0_{-1} v$ (two copies of the doublet top of $\mathcal W(1)$, of charges $\pm 2, 0, 0$: the charges of $\mathcal W(1)$ shifted by $\pm 1$). So $\bar M(\mathbb C) = \mathcal W(0) \to 2\,\mathcal W(1)$, the projective cover $\mathcal P(0)$ modulo its socle:

In [9]:
M0 = M[0]
print([M0.weight_space(d).dimension() for d in range(7)])
print([sum(chi(0, d, g) + chi(1, d - 1, g - 1) + chi(1, d - 1, g + 1) for g in charges) for d in range(7)])
Tg, Wpg, W0g, Wmg = P.gens()
v = M0.highest_weight_vector()
N = M0.submodule([X.mode(v, -1) for X in (Tg, Wpg, W0g, Wmg)])
J = M0.maximal_submodule()
print([N.weight_space(d).dimension() for d in range(7)], [J.weight_space(d).dimension() for d in range(7)])
print(all(N.weight_space(d, horizontal_grade=g).dimension() == chi(1, d - 1, g - 1) + chi(1, d - 1, g + 1)
          for d in range(1, 7) for g in charges))

[1, 4, 5, 8, 13, 20, 34]
[1, 4, 5, 8, 13, 20, 34]
[0, 4, 4, 4, 8, 12, 24] [0, 4, 4, 4, 8, 12, 24]
True


## 6. The projective cover $\mathcal P(0)$

From the Jordan block ($[T]$ acting on the top by $\begin{pmatrix} 0 & 1 \\ 0 & 0\end{pmatrix}$): $\bar M(U)$ has the character $2\chi_{\mathcal W(0)} + 2\chi_{\mathcal W(1)}$ of $\mathcal P(0) = \mathcal W(0) \to 2\,\mathcal W(1) \to \mathcal W(0)$; $L_0 = T_0$ is not diagonalizable on the top; $L_0$ of the top generates the socle $\mathcal W(0)$, and the quotient by it is the module of section 5:

In [10]:
Mj = P.induced_module_up_to(jordan, 8, 5)
print([Mj.weight_space(d).dimension() for d in range(6)])
print(all(Mj.weight_space(d, horizontal_grade=g).dimension() == 2*chi(0, d, g) + chi(1, d - 1, g - 1) + chi(1, d - 1, g + 1)
          for d in range(6) for g in charges))
top = Mj.weight_space(0).basis()
print(top, [Tg.mode(u, 0) for u in top])
soc = Mj.submodule([Tg.mode(top[1], 0)])
print(all(soc.weight_space(d, horizontal_grade=g).dimension() == chi(0, d, g) for d in range(6) for g in charges))
print([(Mj / soc).weight_space(d).dimension() for d in range(6)], [M0.weight_space(d).dimension() for d in range(6)])

[2, 4, 6, 12, 18, 28]
True
[u[0], u[1]] [0, u[0]]
True
[1, 4, 5, 8, 13, 20] [1, 4, 5, 8, 13, 20]


The same module from the symplectic fermions themselves: the module induced over them from the Grassmann algebra of the zero modes $\chi^\pm_0$ (top level $\mathbb C^{2|2}$: $1, \theta^+, \theta^-, \theta^+\theta^-$ of charges $0, 1, -1, 0$) is logarithmic, and its even part has the character of $\mathcal P(0)$, charge by charge:

In [11]:
Ep = matrix(QQ, 4, 4, {(1, 0): 1, (3, 2): 1})
Em = matrix(QQ, 4, 4, {(2, 0): 1, (3, 1): -1})
Msf = SF.induced_module(SF.zhu_algebra().module({'cp': Ep, 'cm': Em}, parities=[0, 1, 1, 0], horizontal_grades=[0, 1, -1, 0]))
print(all(sum(1 for k in Msf.weight_space(d, horizontal_grade=g).basis_keys() if Msf._parity_key(k) == 0)
          == Mj.weight_space(d, horizontal_grade=g).dimension() for d in range(6) for g in charges))

True


## 7. Up to a weight and a depth

`induced_module_up_to(U, max_weight, max_depth)` only uses the relations of weight $\le$ `max_weight` and the depths $\le$ `max_depth`; what it adds always vanishes in $\bar M(U)$, so it maps onto $\bar M(U)$ in these depths, and it can only shrink as `max_weight` and `max_depth` grow. Here nothing changes from the relations of weight 6 (the lowest) and from depth 4 on:

In [12]:
for W, D in ((6, 4), (6, 5), (8, 4), (10, 5)):
    N = P.induced_module_up_to(tops[0], W, D)
    print(W, D, [N.weight_space(d).dimension() for d in range(D + 1)])

6 4 [1, 4, 5, 8, 13]


6 5 [1, 4, 5, 8, 13, 20]
8 4 [1, 4, 5, 8, 13]


10 5 [1, 4, 5, 8, 13, 20]


## 8. The projective cover $\mathcal P(1)$ through the level-1 Zhu algebra

$\mathcal P(1) = \mathcal W(1) \to 2\,\mathcal W(0) \to \mathcal W(1)$ is not generated by its lowest weight space (weight 0, in the middle layer), so it is no $\bar M(U)$ of section 6. Its top layer $\mathcal W(1)$ sits at depth 1, and Dong–Li–Mason's level-$n$ Zhu algebra $A_n$ sees the weights $0, \dots, n$ of a module at once: their universal module $\bar M_n(U)$, induced from a module $U$ placed in degree $n$, may lower as well as raise. Here $n = 1$.

$A_1(\mathcal W(2))$ from the presentation, with the relations up to weight 8 and the filtration bounded by 18 (this algebra maps onto $A_1$ of $\mathcal W(2)$; it is finite dimensional, of dimension 24):

In [13]:
Z1 = P.zhu_algebra_up_to(8, level=1, degree_bound=18)
print(Z1.dimension())

24


Its modules come from its structure (a finite-dimensional algebra over $\mathbb Q$): the radical (the kernel of the trace form), primitive idempotents $e$ (split from minimal polynomials), the projective indecomposable modules $A_1 e$ and the simple modules $A_1 e / J e$, with homogeneous bases for the charge grading. Each simple module is listed with its $[T]$, its dimension and the dimension of its projective cover; the Cartan matrix $c_{ij} = \dim e_i A_1 e_j$; and $\sum \dim S \cdot \dim P = \dim A_1$:

In [14]:
print(len(Z1.radical_basis()))
Ps, Ss = Z1.projective_indecomposable_modules(), Z1.simple_modules()
print([(S_.representation_matrix(Tg)[0, 0], S_.dimension(), P_.dimension()) for P_, S_ in zip(Ps, Ss)])
print(Z1.cartan_invariants_matrix())
print(sum(S_.dimension() * P_.dimension() for P_, S_ in zip(Ps, Ss)))

5


[(2, 2, 2), (11/8, 2, 2), (7/8, 1, 1), (3/8, 2, 2), (-1/8, 1, 1), (1, 2, 4), (0, 1, 2)]


[1 0 0 0 0 0 0]
[0 1 0 0 0 0 0]
[0 0 1 0 0 0 0]
[0 0 0 1 0 0 0]
[0 0 0 0 1 0 0]
[0 0 0 0 0 2 0]
[0 0 0 0 0 0 2]
24


Seven simple modules: the tops of $\mathcal W(-\frac18)$, $\mathcal W(0)$, $\mathcal W(\frac38)$, $\mathcal W(1)$ (weights $h$) and the degree-1 spaces of $\mathcal W(-\frac18)$, $\mathcal W(\frac38)$, $\mathcal W(1)$ (weights $h + 1$; that of $\mathcal W(0)$ is zero). Only $[T] = 0$ and $[T] = 1$ have non-trivial projective covers. On the one of $[T] = 1$ (dimension 4), $[T]$ has two Jordan blocks; its charges are those of the words of $A_1$ (fixed only up to a shift):

In [15]:
P1top = [P_ for P_, S_ in zip(Ps, Ss) if S_.representation_matrix(Tg)[0, 0] == 1][0]
print(P1top.representation_matrix(Tg).jordan_form(), P1top.horizontal_grades())

[1 1|0 0]
[0 1|0 0]
[---+---]
[0 0|1 1]
[0 0|0 1] [(-2), (-2), (0), (0)]


Induced from degree 1 (top weight $1 - 1 = 0$), to depth 4: this is $\mathcal P(1)$, with character $2\chi_1 + 2\chi_0$, and charge by charge (shifted by 1) the two $\mathcal W(0)$ layers carry the charges shifted by $\pm 1$:

In [16]:
D = 4
M1 = P.induced_module_up_to(P1top, 8, D)
print(M1._cover_module.top_weight(), [M1.weight_space(d).dimension() for d in range(D + 1)])


def p1(d, g):
    """The dimension of P(1) at weight d and charge g."""
    return 2*chi(1, d - 1, g) + chi(0, d, g - 1) + chi(0, d, g + 1)


print(all(M1.weight_space(d, horizontal_grade=(g - 1,)).dimension() == p1(d, g) for d in range(D + 1) for g in charges))

0 [2, 4, 6, 12, 18]
True


The Loewy layers: the submodule generated by the weight-0 space is the radical $2\,\mathcal W(0) \to \mathcal W(1)$, the image of $L_0 - 1$ on weight 1 generates the socle $\mathcal W(1)$, and the quotient by the radical is the top $\mathcal W(1)$:

In [17]:
rad = M1.submodule(list(M1.weight_space(0).basis()))
soc = M1.submodule([x for x in (Tg.nproduct(u, 1) - u for u in M1.weight_space(1).basis()) if x])
print([rad.weight_space(d).dimension() for d in range(D + 1)], [soc.weight_space(d).dimension() for d in range(D + 1)],
      [(M1 / rad).weight_space(d).dimension() for d in range(D + 1)])
print(all(rad.weight_space(d, horizontal_grade=(g - 1,)).dimension() == chi(1, d - 1, g) + chi(0, d, g - 1) + chi(0, d, g + 1)
          and soc.weight_space(d, horizontal_grade=(g - 1,)).dimension() == chi(1, d - 1, g)
          for d in range(D + 1) for g in charges))

[2, 2, 4, 10, 14] [0, 2, 2, 2, 4] [0, 2, 2, 2, 4]


True


## 9. Fusion products

`M.fusion_product(N)` is the fusion product $M \boxtimes N$, computed by the Nahm–Gaberdiel–Kausch algorithm (with $M$ at $1$ and $N$ at $0$, as in Kanade–Ridout, arXiv:1812.10713). What it computes is the depth-$d$ quotient $(M \boxtimes N)^{(d)}$: $M \boxtimes N$ modulo the words of modes of total degree $> d$, a finite-dimensional space on which $L_0$ acts (Jordan blocks: logarithmic fusion). It is computed from the tensors of total depth $\le$ `max_depth`; the result maps onto the true quotient once Nahm's special subspace of $M$ fits in that window, and it only shrinks as the window grows.

The depth-0 quotients of the products of the four simple modules, with the generalised eigenvalues of $L_0$ and the sizes of its Jordan blocks:

In [18]:
Wsimple = {h: (M[h].simple_quotient() if h == 0 else M[h]) for h in tops}
hs = [0, 1, -1/8, 3/8]
for i, h1 in enumerate(hs):
    for h2 in hs[i:]:
        F = Wsimple[h1].fusion_product(Wsimple[h2]).depth_quotient_up_to(0, max_depth=3)
        print('W(%s) x W(%s):' % (h1, h2), F.conformal_weights())

W(0) x W(0): [(0, [1])]
W(0) x W(1): [(1, [1, 1])]
W(0) x W(-1/8): [(-1/8, [1])]
W(0) x W(3/8): [(3/8, [1, 1])]
W(1) x W(1): [(0, [1])]
W(1) x W(-1/8): [(3/8, [1, 1])]
W(1) x W(3/8): [(-1/8, [1])]
W(-1/8) x W(-1/8): [(0, [2])]
W(-1/8) x W(3/8): [(0, [1, 1]), (1, [1, 1])]
W(3/8) x W(3/8): [(0, [2])]


This is Gaberdiel–Kausch's table: $\mathcal W(0)$ is the unit; $\mathcal W(1) \boxtimes \mathcal W(1) = \mathcal W(0)$; $\mathcal W(1)$ exchanges $\mathcal W(-\frac18)$ and $\mathcal W(\frac38)$; $\mathcal W(-\frac18)^{\boxtimes 2}$ and $\mathcal W(\frac38)^{\boxtimes 2}$ have a Jordan block of $L_0$ at weight 0 (their fusion product is the logarithmic $\mathcal P(0)$); and $\mathcal W(-\frac18) \boxtimes \mathcal W(\frac38)$ has the two weights 0 and the two weights 1 of $\mathcal P(1)$.

At depth 1, $\mathcal W(-\frac18) \boxtimes \mathcal W(\frac38)$ shows the Jordan blocks of $L_0$ at weight 1 of $\mathcal P(1)$ (between its top and its socle $\mathcal W(1)$); the window is large enough from `max_depth` 2 on (with `max_depth` 1 it does not yet contain the special subspace of $\mathcal W(-\frac18)$, and the result is too small):

In [19]:
P1fus = Wsimple[-1/8].fusion_product(Wsimple[3/8])
print([P1fus.depth_quotient_up_to(1, max_depth=K).dimension() for K in (1, 2, 3, 4)])
print(P1fus.depth_quotient_up_to(1, max_depth=4).conformal_weights())

[6, 8, 8, 8]
[(0, [1, 1]), (1, [2, 2]), (2, [1, 1])]


The depth-0 quotient is a module over the Zhu algebra (its zero modes satisfy the relations of $A(\mathcal W(2))$). For $\mathcal W(-\frac18) \boxtimes \mathcal W(-\frac18)$ it is the Jordan top of section 6, and the module induced from it is $\mathcal P(0)$, charge by charge:

In [20]:
Ffus = Wsimple[-1/8].fusion_product(Wsimple[-1/8]).depth_quotient_up_to(0, max_depth=3)
Ufus = Ffus.zhu_algebra_module(Z)
print(Ufus.generator_matrices()['T'])
Mfus = P.induced_module_up_to(Ufus, 8, 5)
print([Mfus.weight_space(d).dimension() for d in range(6)])
print(all(Mfus.weight_space(d, horizontal_grade=g).dimension() == Mj.weight_space(d, horizontal_grade=g).dimension()
          for d in range(6) for g in charges))

[ -1/4 -1/16]
[    1   1/4]


[2, 4, 6, 12, 18, 28]
True


## 10. The same, by the library: simple modules, projective covers, Loewy layers, extensions
Sections 3–8 found the tops by solving the Gröbner basis on matrices and picked the projective modules of $A_0$, $A_1$ by hand. The library does all of this itself (proofs in the companion paper, in preparation).

`P.simple_modules_up_to(max_weight, max_depth)`: the simple quotients of the modules induced from the simple modules of the computed Zhu algebra $A_0$ (it maps onto $A(\mathcal W(2))$, so every simple module is among them), sorted by lowest weight:

In [21]:
Simples = P.simple_modules_up_to(8, 5)
for S_ in Simples:
    h = S_._weight_base()
    print(h, [S_.weight_space(h + d).dimension() for d in range(6)],
          all(S_.weight_space(h + d, horizontal_grade=(g,)).dimension() == chi(h, d, g) for d in range(6) for g in charges)
          if h in (0, -1/8) else '')
Lm8, L0, L38, L1 = Simples

-1/8 [1, 1, 4, 5, 9, 13] True
0 [1, 0, 1, 4, 5, 8] True
3/8 [2, 2, 4, 6, 12, 16] 
1 [2, 2, 2, 4, 6, 12] 


(For $\mathcal W(1)$ and $\mathcal W(\frac38)$ the library's charges of the top are $0, -2$ rather than $\pm 1$: the grading of a module is fixed only up to a shift.)

`L.projective_cover(max_weight, max_depth)` returns the projective cover $\pi: \mathcal P \to L$. It is $\bar M_n(A_n e)$, induced in degree $n$ from the projective indecomposable module of the level-$n$ Zhu algebra whose head is the top of $L$. The level is chosen so that every simple module congruent to $L$ has lowest weight $\ge h_L - n$ (from the simple modules of $A_0$): $n = 0$ for $\mathcal W(0)$, $n = 1$ for $\mathcal W(1)$ ($\mathcal W(0)$ lies below it). The projective simple modules are their own projective covers:

In [22]:
import time
t = time.time()
pi0, pi1 = L0.projective_cover(8, 4), L1.projective_cover(8, 4)
P0, P1 = pi0.domain(), pi1.domain()
print('%.0f s' % (time.time() - t))
print([P0.weight_space(d).dimension() for d in range(5)], [Mj.weight_space(d).dimension() for d in range(5)])
print([P1.weight_space(d).dimension() for d in range(5)], [M1.weight_space(d).dimension() for d in range(5)])
Pm8 = Lm8.projective_cover(8, 4).domain()
print([Pm8.weight_space(-1/8 + d).dimension() for d in range(5)], [Lm8.weight_space(-1/8 + d).dimension() for d in range(5)])

13 s
[2, 4, 6, 12, 18] [2, 4, 6, 12, 18]
[2, 4, 6, 12, 18] [2, 4, 6, 12, 18]


[1, 1, 4, 5, 9] [1, 1, 4, 5, 9]


Loewy layers: `radical_series_up_to(depth)` (head first: $\operatorname{rad} X$ is the intersection of the kernels of all maps $X \to$ a simple module) and `socle_series_up_to(depth)` (socle first), as multiplicities of the simple modules that can occur (here $\mathcal W(0), \mathcal W(1)$). Both give $\mathcal P(0) = \mathcal W(0) \to 2\,\mathcal W(1) \to \mathcal W(0)$ and $\mathcal P(1) = \mathcal W(1) \to 2\,\mathcal W(0) \to \mathcal W(1)$:

In [23]:
print(P0.radical_series_up_to(4), P0.socle_series_up_to(4))
print(P1.radical_series_up_to(4), P1.socle_series_up_to(4))

[[1, 0], [0, 2], [1, 0]] [[1, 0], [0, 2], [1, 0]]


[[0, 1], [2, 0], [0, 1]] [[0, 1], [2, 0], [0, 1]]


$\operatorname{Ext}^1$ from the projective covers: `M.ext1_up_to(N, depth, cover)` is $\operatorname{Hom}(K, N)$ modulo the restrictions of $\operatorname{Hom}(\mathcal P, N)$, $K = \ker \pi$, computed up to `depth`. Between $\mathcal W(0)$ and $\mathcal W(1)$ it is 2 in both directions and 0 otherwise (the multiplicities of the second Loewy layers); $\mathcal W(-\frac18)$ is projective:

In [24]:
for L, pi in ((L0, pi0), (L1, pi1)):
    print([L.ext1_up_to(N_, 3, pi).dimension() for N_ in (L0, L1)])
print([Lm8.ext1_up_to(N_, 3, Lm8.projective_cover(8, 4)).dimension() for N_ in (Lm8, L38)])

[0, 2]


[2, 0]
[0, 0]


Each class gives its extension module, `extension(c)`: the pushout of the cover along the map $g: K \to N$ of the class. Extensions of $\mathcal W(0)$ by $\mathcal W(1)$ and of $\mathcal W(1)$ by $\mathcal W(0)$, non-split (socle $\mathcal W(1)$, respectively $\mathcal W(0)$, alone):

In [25]:
E01 = L0.ext1_up_to(L1, 3, pi0).extension([1, 0])
E10 = L1.ext1_up_to(L0, 3, pi1).extension([1, 0])
for E in (E01, E10):
    print([E.weight_space(d).dimension() for d in range(4)], E.socle_series_up_to(3))

[1, 2, 3, 6] [[0, 1], [1, 0]]
[1, 2, 3, 6] [[1, 0], [0, 1]]


The cover need not be projective. For $\bar M_n(U) \to M$ there is the exact sequence
$$0 \to \operatorname{Hom}(M, N) \to \operatorname{Hom}(\bar M_n(U), N) \to \operatorname{Hom}(K, N) \to \operatorname{Ext}^1(M, N) \xrightarrow{\ \delta\ } \operatorname{Ext}^1_{A_n}(U, N(n)):$$
the classes whose degree-$n$ part splits, plus those classes of the Zhu algebra level that lift (`delta_image()`: those for which some $g: \tilde K \to N$ restricts to the inclusion of $N(n)$, $\tilde K$ the kernel of the module induced from the extension of $U$). The Jordan block of $[T]$ at 0 is a self-extension of the trivial $A_0$-module. It does not lift to a self-extension of $\mathcal W(0)$ ($\operatorname{Ext}^1(\mathcal W(0), \mathcal W(0)) = 0$), but it lifts to an extension of $\bar M(\mathbb C) = \mathcal W(0) \to 2\,\mathcal W(1)$ by $\mathcal W(0)$, and that extension is $\mathcal P(0)$:

In [26]:
W0s = M0.simple_quotient()
E = W0s.ext1_up_to(W0s, 3, Hom(M0, W0s).basis()[0])
print(E.top_level_dimension(), E.delta_image(), E.dimension())
F = M0.ext1_up_to(W0s, 3, End(M0).identity())
print(F.top_level_dimension(), F.delta_image(), F.dimension())
Pd = F.top_extension([1])
print([Pd.weight_space(d).dimension() for d in range(4)], Pd.radical_series_up_to(3))

1 [] 0
1 [(1,)] 1
[2, 4, 6, 12] [[1, 0], [0, 2], [1, 0]]


Indecomposable modules are quotients of projective ones and can be fused. $\mathcal P(0) / \operatorname{rad}^2 \mathcal P(0) = \mathcal W(0) \to 2\,\mathcal W(1)$ (character $\chi_0 + 2\chi_1$); with the projective $\mathcal W(-\frac18)$ it gives $\mathcal W(-\frac18) \oplus 2\,\mathcal W(\frac38)$, $\mathcal P(0)$ gives twice that, and the depth-0 quotient of $\mathcal P(0) \boxtimes \mathcal P(0)$ shows the tops of $2\,\mathcal P(0) \oplus 2\,\mathcal P(1)$ (two Jordan blocks at weight 0):

In [27]:
subs, layers = P0.radical_series_up_to(4, submodules=True)
R0 = P0.quotient(subs[2])
print([R0.weight_space(d).dimension() for d in range(5)])
for X, Y_ in ((R0, Lm8), (P0, Lm8), (P0, P0)):
    print(X.fusion_product(Y_).depth_quotient_up_to(0, 3).conformal_weights())

[1, 4, 5, 8, 13]
[(-1/8, [1]), (3/8, [1, 1, 1, 1])]


[(-1/8, [1, 1]), (3/8, [1, 1, 1, 1])]
[(0, [2, 2, 1, 1, 1, 1]), (1, [1, 1, 1, 1])]


## 11. Not there yet

* $\mathcal W(p)$ for $p \ge 3$ (a subalgebra of the lattice vertex algebra of $\sqrt{2p}\,\mathbb Z$) and the $(p_+, p_-)$ triplet algebras (`14_triplet_algebra.ipynb`), in particular the projective cover of the vacuum module of the $(2, 3)$ triplet algebra at $c = 0$ (Gaberdiel–Runkel–Wood): they need larger depths, i.e. faster module actions (the C++ engines) and parallel linear algebra.
* Twisted (Ramond) induced modules: here the twisted sector of the symplectic fermions is reached through $\mathcal W(2)$'s own untwisted modules $\mathcal W(-\frac18)$, $\mathcal W(\frac38)$.

## References

- [Kau1991] H. G. Kausch, *Extended conformal algebras generated by a multiplet of primary fields*, Phys. Lett. B 259 (1991), 448–455, [doi:10.1016/0370-2693(91)91655-F](https://doi.org/10.1016/0370-2693(91)91655-F).
- [GK1996b] M. R. Gaberdiel, H. G. Kausch, *A rational logarithmic conformal field theory*, Phys. Lett. B 386 (1996), 131–137, [arXiv:hep-th/9606050](https://arxiv.org/abs/hep-th/9606050). — the triplet algebra $\mathcal W(2)$ and its modules.
- [DLM1998] C. Dong, H. Li, G. Mason, *Vertex operator algebras and associative algebras*, J. Algebra 206 (1998), 67–96, [arXiv:q-alg/9612010](https://arxiv.org/abs/q-alg/9612010). — the universal modules $\bar M(U)$.
- [Nah1994] W. Nahm, *Quasi-rational fusion products*, Internat. J. Modern Phys. B 8 (1994), 3693–3702, [arXiv:hep-th/9402039](https://arxiv.org/abs/hep-th/9402039).
- [GK1996] M. R. Gaberdiel, H. G. Kausch, *Indecomposable fusion products*, Nuclear Phys. B 477 (1996), 293–318, [arXiv:hep-th/9604026](https://arxiv.org/abs/hep-th/9604026). — the Nahm–Gaberdiel–Kausch algorithm and the fusion rules of $\mathcal W(2)$.
- [KR2019] S. Kanade, D. Ridout, *NGK and HLZ: fusion for physicists and mathematicians*, in: Affine, vertex and W-algebras, Springer INdAM Ser. 37 (2019), 135–181, [arXiv:1812.10713](https://arxiv.org/abs/1812.10713). — the conventions of the fusion computation.